# Catalog streaming from hugging face

- **Bandwidth-bound at Hugging Face, not compute-bound.** Full materialization through lsdb runs at 28.5 MB/s against 32.0 MB/s for a raw byte read of the same file — parquet decode, pd.concat, graph building and serialization together cost 11%. There's nothing worth optimizing on our side of the pipe.

- **We sustain 36.6 MB/s with two workers,** which is ~28 hours for one epoch over the full 3.71 TB catalog. The _10arcs sibling catalog is the same sky in 0.12 TB — about 50 minutes per epoch at the same rate.

- **Concurrency scales and we're not near a ceiling.** Two workers gave 1.63× one worker, so we're limited per connection rather than by a bandwidth cap. Adding workers is the direct lever; 4 and 8 are the next points to measure.

- **21.6% of every cycle is dead time we can recover.** InfiniteStream shuffles each chunk's rows before submitting the next one, so the workers sit idle for ~9 s per chunk. The two operations have no data dependency — reordering them upstream takes us to 46.6 MB/s, a 28% gain. (Caveat: it needs split RNG streams to stay reproducible, since shuffle and partition selection share one generator.)

- **Measure this catalog in MB/s, never rows/s.** The image.flux cutout column is 99.9% of the bytes at ~273 KB/row, so row counts are meaningless as a throughput unit — and this matters for the crossmatch comparison, where rows/s will fall even if throughput is flat.

- **Clean run, but a thin one.** Zero rate-limit errors across the whole benchmark. Against that: four steady chunks is a small sample, and HF's own throughput varied 1.6× between our two measurement sessions — so cross-session comparisons need the raw-read control as a normalizer.

## Set up

In [1]:
# Kernel smoke test -- run this FIRST after a restart.
# If it doesn't return instantly, the kernel isn't really ready and the problem is
# the Jupyter connection, not the dask/lsdb imports below.
import sys

print("kernel alive:", sys.version.split()[0])

kernel alive: 3.13.13


In [2]:
# Make a dask client; set up dashboard for slacd

import os
import socket
from dask.distributed import Client


client = Client(
    n_workers=2,
    threads_per_worker=1,
    memory_limit="32GB",
)

# username = os.getenv("USER")
# current_compute_node = socket.gethostname()
# dashboard_port = client.scheduler_info()["services"]["dashboard"]

# print(f"Run this in a local shell:")
# print(
#     f"  ssh -L 8787:{current_compute_node}:{dashboard_port} -J {username}@s3dflogin.slac.stanford.edu {username}@{current_compute_node}"
# )
# print(f"\nThen open: http://localhost:8787/status")


In [3]:
from time import perf_counter

import fsspec
import pandas as pd
from dask.distributed import performance_report, wait

import lsdb
from lsdb.streams import InfiniteStream

In [4]:
!mkdir -p reports/

## Initial: Single catalog

In [5]:
cat = lsdb.open_catalog(
    "hf://datasets/UniverseTBD/mmu_ssl_legacysurvey_north/"
)
cat.npartitions

5488

## Benchmark: phase-split timing of an `InfiniteStream`

`cProfile` has been removed from this benchmark. Two reasons:

1. **It answers the wrong question.** This is an IO-bound pipeline with three *known*
   phases, so what we need is wall-clock measurement of named regions, not
   function-level discovery of unexpected CPU.
2. **On Python 3.12+ it is not thread-scoped.** `cProfile` is built on `sys.monitoring`,
   which is interpreter-wide, so the `LocalCluster` scheduler thread and the bokeh
   dashboard thread land in the same flat call graph as the main thread. The previous
   run showed the damage plainly: 904 s of cumtime inside a 419 s window, `epoll.poll`
   with tottime 117.7 s against cumtime 0.09 s, and `sample` credited 22.5 s while its
   own callees summed to ~0.1 s. **No magnitude from that profile is trustworthy** --
   including the ones that looked plausible.

### What `__next__` actually does

The submit for chunk *k+1* happens inside the `next()` call that returns chunk *k*, and
the very first submit happens at `iter()` time, not at the first `next()`. So every
iteration is three phases:

| Phase | Call | Runs on |
| --- | --- | --- |
| A. wait | `self.future.result()` | blocked; work is on the workers |
| B. shuffle | `result.sample(frac=1, ...)` | main thread, single-threaded |
| C. submit | `submit_next_partitions(...)` | main thread |

`dask.distributed.wait()` blocks on a future *without consuming it*, so bracketing it
just before `next()` splits phase A from B+C at no cost -- we would have been blocked
inside `result()` for that same interval anyway. The stream's behaviour, submit order
and RNG draws are all unchanged.

### Reading the numbers

- `wait_s` is **unhidden worker-side latency**, not network time alone: HTTP fetch,
  parquet decode, the worker's `pd.concat`, and result serialization. The IO baseline
  section below separates fetch from decode.
- With a no-op consumer, `wait_s` *is* the full worker time, because nothing overlaps
  with the prefetch. That identity breaks the moment a real op is added, which is
  exactly why this no-op run is the baseline worth recording carefully.
- `post_s` (B + C) is **dead time, not overlapped work**: the submit happens *after* the
  shuffle, so the workers sit idle with nothing queued while the shuffle runs.
- Chunk 0 is not steady-state. It absorbs catalog metadata loading and TLS setup and had
  no prefetch behind it. It is recorded, then excluded from the summary.

Close the dashboard browser tab while timing. A connected tab pushes websocket updates
for the entire run -- it accounted for 72 s of event-loop churn in the previous profile.
`performance_report` collects the same diagnostics with no browser attached.

In [6]:
N_ITERS = 5
PARTITIONS_PER_CHUNK = 2
SEED = 1

# Measured from this catalog's parquet footers: one row group is 50.74 MB compressed
# against 56.62 MB uncompressed. The image floats barely compress, so decoded bytes are
# a good proxy for bytes on the wire -- good to ~11%, which is enough to report MB/s.
WIRE_RATIO = 0.896


def run_stream_benchmark(catalog, client, n_iters, partitions_per_chunk, seed=SEED):
    """Pull `n_iters` chunks, timing each iteration's phases separately.

    Returns one row per chunk:
      wait_s  -- blocked on the future (phase A); unhidden worker-side latency
      post_s  -- shuffle + submit (phases B and C); main-thread dead time
      nbytes  -- decoded in-memory size of the chunk
    """
    stream = InfiniteStream(
        catalog=catalog,
        client=client,
        partitions_per_chunk=partitions_per_chunk,
        seed=seed,
    )
    # iter() submits chunk 0, so chunk 0's clock effectively starts on this line.
    stream_iter = iter(stream)

    records = []
    for k in range(n_iters):
        t0 = perf_counter()
        wait(stream_iter.future)  # blocks without consuming the future
        t1 = perf_counter()
        chunk = next(stream_iter)  # result() already satisfied; shuffle, then submit
        t2 = perf_counter()

        records.append(
            {
                "chunk": k,
                "wait_s": t1 - t0,
                "post_s": t2 - t1,
                "rows": len(chunk),
                # deep=True accounts the Arrow buffers behind the nested `image`
                # column, so this is real bytes rather than a pointer count.
                "nbytes": int(chunk.memory_usage(deep=True).sum()),
            }
        )
        # Drop the chunk before the next one lands. Without this, two ~1.7 GB frames
        # are briefly alive at once at the moment next() rebinds the name.
        del chunk

    return pd.DataFrame(records)

In [7]:
def summarize(results, label=""):
    """Print steady-state throughput, excluding the cold chunk 0."""
    steady = results[results["chunk"] > 0]
    mb = steady["nbytes"] / 1e6
    total_s = (steady["wait_s"] + steady["post_s"]).sum()
    n = len(steady)

    print(f"{label}{n} chunks (chunk 0 excluded as cold)")
    print(f"  decoded       {mb.sum():8.1f} MB total, {mb.mean():7.1f} MB/chunk")
    print(f"  wall          {total_s:8.1f} s total, {total_s / n:7.2f} s/chunk")
    print(
        f"  wait_s        {steady['wait_s'].mean():7.2f} s mean"
        f"  [{steady['wait_s'].min():.2f}, {steady['wait_s'].max():.2f}]"
    )
    print(
        f"  post_s        {steady['post_s'].mean():7.2f} s mean"
        f"  ({100 * steady['post_s'].sum() / total_s:.1f}% of wall, not overlapped)"
    )
    print(f"  decoded MB/s  {mb.sum() / total_s:7.2f}")
    print(f"  wire MB/s     {WIRE_RATIO * mb.sum() / total_s:7.2f}  (approx)")
    print(f"  rows/s        {steady['rows'].sum() / total_s:7.1f}")

In [8]:
REPORT_PATH = "reports/dask-report.html"
TIMINGS_PATH = "reports/stream_infinite_timings.csv"

# performance_report renders its HTML on context exit, outside the per-chunk brackets,
# so it does not pollute the timings it wraps. It gives us the half that client-side
# timing cannot see: per-task durations, deserialization cost, worker memory and spills.
with performance_report(filename=REPORT_PATH):
    results = run_stream_benchmark(cat, client, N_ITERS, PARTITIONS_PER_CHUNK)

results.to_csv(TIMINGS_PATH, index=False)
print(f"wrote {TIMINGS_PATH} and {REPORT_PATH}")
results

wrote reports/stream_infinite_timings.csv and reports/dask-report.html


,chunk,wait_s,post_s,rows,nbytes
0,0,27.068353,8.599115,4423,1235256323
1,1,21.075353,8.249325,4379,1222968159
2,2,64.973891,19.381076,10479,2926577358
3,3,29.095833,5.298022,5556,1551680741
4,4,19.064209,4.083203,4596,1283571776


In [9]:
print(
    f"cold chunk 0: wait {results.loc[0, 'wait_s']:.1f}s"
    f"  post {results.loc[0, 'post_s']:.2f}s\n"
)
summarize(results)

cold chunk 0: wait 27.1s  post 8.60s

4 chunks (chunk 0 excluded as cold)
  decoded         6984.8 MB total,  1746.2 MB/chunk
  wall             171.2 s total,   42.81 s/chunk
  wait_s          33.55 s mean  [19.06, 64.97]
  post_s           9.25 s mean  (21.6% of wall, not overlapped)
  decoded MB/s    40.79
  wire MB/s       36.55  (approx)
  rows/s          146.1


## Baseline: is the bottleneck the network or the decode?

`wait_s` bundles fetch, parquet decode, `pd.concat` and serialization together. These two
cells pull them apart by comparing a raw byte read against a full materialization of the
same kind of file, both on a single connection.

Compare the raw read's MB/s against the **wire MB/s** line from `summarize` (not the
decoded line -- the raw read is of compressed bytes):

- **close to raw** -> bandwidth-bound at Hugging Face; lsdb, dask and parquet are not the cost
- **well below raw** -> decode / deserialization is a real term worth attacking

This also gives the first two points of a concurrency curve for free. The main benchmark
runs 2 partitions in parallel; if that comes in near 2x the single-connection rate,
scaling is linear and we are per-connection limited rather than sitting against a
bandwidth ceiling.

One measurement note: a cold connection to Hugging Face runs about 10x slower than a warm one (~2.3 MB/s against ~20 MB/s in a spot check), so the raw-read cell discards a warmup read before starting its clock. That single warm sample already hints at something worth confirming -- it is *above* the 17.8 MB/s the previous run achieved across two workers, which would mean the stream is not per-connection bandwidth-limited and the cost is elsewhere.


In [10]:
# A known partition file in this catalog. Only a bounded prefix is read -- enough to
# measure steady-state throughput without pulling the whole file (this one is ~2 GB).
PARTITION_URL = (
    "hf://datasets/UniverseTBD/mmu_ssl_legacysurvey_north/"
    "mmu_ssl_legacysurvey_north/dataset/Norder=4/Dir=0/Npix=1005.parquet"
)
WARMUP_BYTES = 8 * 1024**2
RAW_READ_BYTES = 256 * 1024**2

with fsspec.open(PARTITION_URL).open() as f:
    # The first read pays for redirect resolution to the CDN, the TLS handshake and the
    # initial block fetch. Measured here at ~2.3 MB/s against ~20 MB/s once warm -- a
    # 10x distortion -- so burn it before starting the clock.
    t0 = perf_counter()
    f.read(WARMUP_BYTES)
    warmup_s = perf_counter() - t0

    t0 = perf_counter()
    buf = f.read(RAW_READ_BYTES)
    raw_s = perf_counter() - t0

raw_mb = len(buf) / 1e6
print(f"warmup      {WARMUP_BYTES / 1e6:6.0f} MB in {warmup_s:5.1f}s"
      f" -> {WARMUP_BYTES / 1e6 / warmup_s:6.2f} MB/s  (discarded)")
print(f"raw read    {raw_mb:6.0f} MB in {raw_s:5.1f}s"
      f" -> {raw_mb / raw_s:6.2f} MB/s  (no decode)")
del buf

warmup           8 MB in   1.4s ->   6.21 MB/s  (discarded)
raw read       268 MB in   8.4s ->  32.03 MB/s  (no decode)


In [11]:
# Same stack, one partition per chunk, so a single worker fetches and decodes at a time.
single = run_stream_benchmark(cat, client, n_iters=3, partitions_per_chunk=1, seed=7)
summarize(single, label="1 partition/chunk: ")
single

1 partition/chunk: 2 chunks (chunk 0 excluded as cold)
  decoded         1294.7 MB total,   647.4 MB/chunk
  wall              44.8 s total,   22.39 s/chunk
  wait_s          20.33 s mean  [18.55, 22.12]
  post_s           2.06 s mean  (9.2% of wall, not overlapped)
  decoded MB/s    28.91
  wire MB/s       25.90  (approx)
  rows/s          103.5


,chunk,wait_s,post_s,rows,nbytes
0,0,41.505448,2.492381,2556,713840084
1,1,22.123320,2.187689,2476,691497733
2,2,18.546225,1.929893,2160,603245331


## Smoke check: did the workers hit HTTP trouble?

Hugging Face rate-limits. A 429 or a connection reset that gets retried would show up
only as an inflated `wait_s`, so it is worth a quick look. Note the limitation: this only
catches what the workers actually *logged*. Retries swallowed inside `aiohttp` or
`huggingface_hub` are invisible here, so no hits is weak evidence, not proof.

On caching: `HfFileSystem._open` returns an `HfFileSystemFile` doing ranged HTTP with
fsspec's in-memory per-handle block cache. There is **no local disk cache**, so nothing
carries between chunks and these throughput numbers are not inflated by re-reads.
(`HfFileSystem.cachable = True` caches the *filesystem instance*, not the data.)

In [12]:
NOISY = ("429", "Retry", "retry", "TooManyRequests", "ConnectionReset", "Timeout")

hits = [
    (addr, level, msg)
    for addr, entries in client.get_worker_logs().items()
    for level, msg in entries
    if any(s in msg for s in NOISY)
]

print(f"{len(hits)} suspicious worker log lines")
for addr, level, msg in hits[:20]:
    print(f"  [{addr}] {level}: {msg[:200]}")

0 suspicious worker log lines
